In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
from numba import njit
from tqdm import tqdm
import subprocess

L = 50
J0 = 0.5
N_STEPS = 15000
N_SNAP = 10

T_c_2d = 2 * J0 / np.log(1 + np.sqrt(2))
print(f"2D Ising T_c = {T_c_2d:.3f} for J0 = {J0}")

TEMPERATURES = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
MU_GRID = np.linspace(-2.5, 0.5, 30)

DATA_DIR = Path("data/conductance")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
def generate_snapshot(T, mu, J0, L, steps, run_id):
    """Run MC simulation and return lattice snapshot as numpy array."""
    out_path = DATA_DIR / f"snap_T{T:.3f}_mu{mu:.3f}_run{run_id}.csv"
    ps_cli = Path("target/release/ps_cli")
    if not ps_cli.exists():
        subprocess.run(["cargo", "build", "--release"], check=True)

    cmd = [
        str(ps_cli),
        "--steps", str(steps),
        "--temperature", str(T),
        f"--chem-potential={mu}",
        f"--interaction={J0}",
        "--width", str(L),
        "--height", str(L),
        "--output", str(DATA_DIR / "tmp.csv"),
        "--snapshot-csv", str(out_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True)

    with open(out_path) as f:
        grid = [list(map(int, line.strip().split(","))) for line in f if line.strip()]
    return np.array(grid, dtype=np.int32)

In [ ]:
@njit(cache=True)
def build_system(lattice):
    """
    Build sparse linear system for Kirchhoff's equations.
    Bond conductance g_ij = n_i * n_j (hopping between occupied sites).
    Left boundary (x=0): V=1, Right boundary (x=L-1): V=0.
    Periodic in y-direction.
    
    Free node indexing: fi = y * (Lx - 2) + (x - 1) for 0 < x < Lx-1.
    """
    Ly, Lx = lattice.shape
    n_free = Ly * (Lx - 2)
    max_nnz = n_free * 5  # at most 4 off-diag + 1 diag per node

    rows = np.empty(max_nnz, dtype=np.int64)
    cols = np.empty(max_nnz, dtype=np.int64)
    vals = np.empty(max_nnz, dtype=np.float64)
    b = np.zeros(n_free, dtype=np.float64)
    diag = np.zeros(n_free, dtype=np.float64)

    nnz = 0

    dy_arr = np.array([-1, 1, 0, 0])
    dx_arr = np.array([0, 0, -1, 1])

    for y in range(Ly):
        for x in range(1, Lx - 1):
            fi = y * (Lx - 2) + (x - 1)
            ni = lattice[y, x]

            for d in range(4):
                ny = (y + dy_arr[d]) % Ly
                nx = x + dx_arr[d]

                if nx < 0 or nx >= Lx:
                    continue

                nj = lattice[ny, nx]
                g = float(ni * nj)
                if g == 0.0:
                    continue

                diag[fi] += g

                if nx == 0:        # left boundary: V = 1
                    b[fi] += g
                elif nx == Lx - 1: # right boundary: V = 0
                    pass
                else:
                    fj = ny * (Lx - 2) + (nx - 1)
                    rows[nnz] = fi
                    cols[nnz] = fj
                    vals[nnz] = -g
                    nnz += 1

    for fi in range(n_free):
        rows[nnz] = fi
        cols[nnz] = fi
        vals[nnz] = diag[fi] + 1e-12
        nnz += 1

    return rows[:nnz], cols[:nnz], vals[:nnz], b, n_free


@njit(cache=True)
def compute_current(lattice, V_free):
    """Total current flowing from left boundary into the lattice."""
    Ly, Lx = lattice.shape
    I_total = 0.0
    for y in range(Ly):
        g = float(lattice[y, 0] * lattice[y, 1])
        if g > 0:
            fj = y * (Lx - 2)  # free index of (y, x=1)
            I_total += g * (1.0 - V_free[fj])
    return I_total


def effective_conductance(lattice):
    rows, cols, vals, b, n_free = build_system(lattice)
    if n_free == 0:
        return 0.0
    A = coo_matrix((vals, (rows, cols)), shape=(n_free, n_free)).tocsr()
    V = spsolve(A, b)
    return compute_current(lattice, V)

_dummy = np.ones((4, 4), dtype=np.int32)
_ = build_system(_dummy)
_ = compute_current(_dummy, np.zeros(8))

In [ ]:
G_max = effective_conductance(np.ones((L, L), dtype=np.int32))
print(f"G_max (fully occupied {L}x{L} lattice) = {G_max:.4f}")
print(f"Theoretical G_max = L/(L-1) = {L/(L-1):.4f}")

results = {T: {"rho": [], "G_norm": []} for T in TEMPERATURES}

total = len(TEMPERATURES) * len(MU_GRID) * N_SNAP
with tqdm(total=total, desc="Computing conductance") as pbar:
    for T in TEMPERATURES:
        for mu in MU_GRID:
            for k in range(N_SNAP):
                lattice = generate_snapshot(T, mu, J0, L, N_STEPS, k)
                rho = lattice.sum() / lattice.size
                G = effective_conductance(lattice) / G_max

                results[T]["rho"].append(rho)
                results[T]["G_norm"].append(G)
                pbar.set_postfix(T=f"{T:.1f}", mu=f"{mu:.2f}", rho=f"{rho:.3f}")
                pbar.update(1)

for T in TEMPERATURES:
    results[T]["rho"] = np.array(results[T]["rho"])
    results[T]["G_norm"] = np.array(results[T]["G_norm"])

In [ ]:
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

fig, ax = plt.subplots(figsize=(7, 4))

rho_line = np.linspace(0, 1, 200)

ax.plot(rho_line, rho_line, '-', color='gray', lw=1, label=r"Mean field ($\sigma \propto \rho$)")

p_c = 0.593
ax.axvline(p_c, color='gray', ls='--', lw=1, alpha=0.7)
ax.text(p_c + 0.01, 0.35, r"$p_c$", color='gray', fontsize=11)

cmap = plt.cm.coolwarm
norm = plt.Normalize(vmin=min(TEMPERATURES), vmax=max(TEMPERATURES))

for T in TEMPERATURES[1:]:
    rho = results[T]["rho"]
    G = results[T]["G_norm"]
    col = cmap(norm(T))

    bin_edges = np.linspace(0, 1, 21)
    bc, bm, bs = [], [], []
    for i in range(len(bin_edges) - 1):
        mask = (rho >= bin_edges[i]) & (rho < bin_edges[i + 1])
        if mask.sum() >= 2:
            bc.append(0.5 * (bin_edges[i] + bin_edges[i + 1]))
            bm.append(G[mask].mean())
            bs.append(G[mask].std())

    bc, bm, bs = np.array(bc), np.array(bm), np.array(bs)

    above_tc = T > T_c_2d + 0.01
    marker = 'o' if above_tc else 's'
    label = rf"$T = {T:.1f}$"
    if abs(T - T_c_2d) < 0.05:
        label += r" ($\approx T_c$)"

    ax.errorbar(bc, bm, yerr=bs, fmt=f'{marker}-', color=col,
                lw=1.5, ms=5, capsize=3, label=label)

ax.set_xlabel(r"Carrier density $\rho$", fontsize=12)
ax.set_ylabel(r"Normalized conductance $G/G_{\mathrm{max}}$", fontsize=12)
ax.legend(frameon=False, loc="upper left", fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure5.pdf")
plt.show()

In [ ]:
from matplotlib.colors import Normalize as mplNorm
from matplotlib.cm import ScalarMappable

def get_voltage_field(lattice):
    """Solve resistor network and return voltage at every site."""
    Ly, Lx = lattice.shape
    V_full = np.full((Ly, Lx), np.nan)
    
    for y in range(Ly):
        if lattice[y, 0] == 1:
            V_full[y, 0] = 1.0
        if lattice[y, Lx - 1] == 1:
            V_full[y, Lx - 1] = 0.0
    
    rows, cols, vals, b, n_free = build_system(lattice)
    if n_free == 0:
        return V_full
    A = coo_matrix((vals, (rows, cols)), shape=(n_free, n_free)).tocsr()
    V_free = spsolve(A, b)
    
    for y in range(Ly):
        for x in range(1, Lx - 1):
            if lattice[y, x] == 1:
                fi = y * (Lx - 2) + (x - 1)
                V_full[y, x] = V_free[fi]
    
    return V_full


cases = [
    {"T": 0.8, "mu": -0.8},
    {"T": 0.8, "mu": -0.2},
    {"T": 0.8, "mu": 0.2},
]

fig, axes = plt.subplots(2, len(cases), figsize=(2.5 * len(cases), 5),
                         gridspec_kw={"height_ratios": [1, 1], "hspace": 0.05})

for j, case in enumerate(tqdm(cases, desc="Generating snapshots")):
    lattice = generate_snapshot(case["T"], case["mu"], J0, L, N_STEPS, run_id=99)
    V_field = get_voltage_field(lattice)
    rho = lattice.sum() / lattice.size
    G = effective_conductance(lattice) / G_max

    ax_top = axes[0, j]
    ax_top.imshow(lattice, origin="lower", cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    ax_top.set_title(f"$T = {case['T']}$, $\\mu = {case['mu']}$, $\\rho = {rho:.2f}$", fontsize=10)
    ax_top.set_xticks([])
    ax_top.set_yticks([])

    ax_bot = axes[1, j]
    bg = np.ones((L, L, 4))
    bg[lattice == 0] = [0.92, 0.92, 0.92, 1.0]
    bg[lattice == 1] = [1, 1, 1, 0]
    ax_bot.imshow(bg, origin="lower", interpolation="nearest")
    masked_V = np.ma.masked_where(lattice == 0, V_field)
    ax_bot.imshow(masked_V, origin="lower", cmap="RdYlBu_r", vmin=0, vmax=1,
                  interpolation="nearest")
    ax_bot.set_title(f"$G/G_{{\\mathrm{{max}}}} = {G:.2f}$", fontsize=10)
    ax_bot.set_xticks([])
    ax_bot.set_yticks([])

pos_bot = axes[1, -1].get_position()
cbar_ax = fig.add_axes([0.92, pos_bot.y0, 0.015, pos_bot.height])
fig.colorbar(ScalarMappable(norm=mplNorm(0, 1), cmap="RdYlBu_r"), cax=cbar_ax, label="Voltage $V$")

fig.tight_layout(rect=[0, 0, 0.90, 1])
fig.savefig(FIG_DIR / "conductance_snapshots.pdf", bbox_inches="tight")
plt.show()